# HStream Extractor (Colab)

Bulk downloader + optional subtitle remuxer for hstream.moe

**Cookies:** Upload a `cookies.txt` (Netscape format) via the left sidebar, then set `COOKIES_FILE` below. Do not paste raw cookie headers.

In [ ]:
import os
import subprocess
import requests
import glob
from tqdm.notebook import tqdm

print("Installing dependencies...")
try:
    subprocess.run(["pip", "install", "--upgrade", "yt-dlp", "requests", "tqdm"], check=True)
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "aria2", "ffmpeg"], check=True)
    print("Dependencies installed successfully!")
except Exception as e:
    print(f"Warning during dependency setup: {e}")

In [ ]:
# @title Settings
URL_LIST = "https://hstream.moe/hentai/sweet-home-h-na-oneesan-wa-suki-desu-ka-1 https://hstream.moe/hentai/sweet-home-h-na-oneesan-wa-suki-desu-ka-2 https://hstream.moe/hentai/sweet-home-h-na-oneesan-wa-suki-desu-ka-3"  #@param {type:"string"}
DESTINATION_FOLDER = "/content/downloads"  #@param {type:"string"}
COOKIES_FILE = ""  #@param {type:"string"}
SERIES_SLUG = ""  #@param {type:"string"}
YEAR = "2024"  #@param {type:"string"}

print("Settings loaded")
print("URLs:", URL_LIST)
print("Cookies file:", COOKIES_FILE or "(none)")

In [ ]:
if not os.path.exists(DESTINATION_FOLDER):
    os.makedirs(DESTINATION_FOLDER)

urls = [u.strip() for u in URL_LIST.replace("\n", " ").split() if u.strip()]
print(f"Found {len(urls)} links to process.\n")

cookies_path = COOKIES_FILE.strip()
if cookies_path and not os.path.exists(cookies_path):
    print(f"WARNING: Cookies file not found: {cookies_path}")
    print("Upload cookies.txt via the left sidebar, then set COOKIES_FILE = cookies.txt")

for index, url in enumerate(tqdm(urls, desc="Overall Progress", unit="video"), start=1):
    tqdm.write(f"\nProcessing [{index}/{len(urls)}]: {url}")

    output_template = os.path.join(DESTINATION_FOLDER, "%(title)s.%(ext)s")

    cmd = [
        "yt-dlp", "-v", "--downloader", "aria2c",
        "--concurrent-fragments", "8",
        "-o", output_template,
    ]
    if cookies_path and os.path.exists(cookies_path):
        cmd += ["--cookies", cookies_path]
    cmd.append(url)

    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as e:
        tqdm.write(f"Error downloading video for {url}: {e}")
        continue

    files = glob.glob(os.path.join(DESTINATION_FOLDER, "*"))
    if not files:
        tqdm.write("No output files found. Skipping...")
        continue

    latest_video = max(files, key=os.path.getctime)
    base_name = os.path.splitext(os.path.basename(latest_video))[0]
    final_mkv = os.path.join(DESTINATION_FOLDER, f"{base_name}.mkv")

    if latest_video == final_mkv:
        continue

    parts = url.rstrip("/").split("/")
    ep_num = parts[-1].split("-")[-1]

    series_name_clean = SERIES_SLUG.strip()
    if not series_name_clean:
        series_name_clean = "-".join(parts[-1].split("-")[:-1]) if "-" in parts[-1] else parts[-1]

    sub_url = f"https://oppai-str.shoujo-h.org/{YEAR}/{series_name_clean}/E{int(ep_num):02d}/eng.ass"
    sub_path = os.path.join(DESTINATION_FOLDER, f"{base_name}.ass")
    tqdm.write("Downloading subtitle...")

    try:
        sub_res = requests.get(sub_url, stream=True, timeout=30)
        if sub_res.status_code == 200:
            total_size = int(sub_res.headers.get("content-length", 0))
            block_size = 1024

            with open(sub_path, "wb") as f, tqdm(
                desc="Subtitle Progress",
                total=total_size,
                unit="B",
                unit_scale=True,
                unit_divisor=1024,
                leave=False
            ) as sub_bar:
                for data in sub_res.iter_content(block_size):
                    sub_bar.update(len(data))
                    f.write(data)

            tqdm.write("Remuxing video and subtitles into MKV...")
            subprocess.run([
                "ffmpeg", "-y", "-i", latest_video, "-i", sub_path,
                "-map", "0", "-map", "1", "-c", "copy",
                "-metadata:s:s:0", "language=eng", final_mkv
            ], check=True)

            if os.path.exists(sub_path):
                os.remove(sub_path)
            if latest_video != final_mkv and os.path.exists(latest_video):
                os.remove(latest_video)

            tqdm.write(f"Successfully Saved: {final_mkv}")
        else:
            tqdm.write(f"Subtitle endpoint returned status {sub_res.status_code}. Keeping original video.")
    except Exception as ex:
        tqdm.write(f"Error processing subtitle/remux: {ex}")

print("\n" + "=" * 50)
print("ALL TASKS COMPLETED!")


In [ ]:
!zip -r /content/hstream_downloads.zip {DESTINATION_FOLDER}
print("Created: /content/hstream_downloads.zip")
print("Download from left sidebar -> Files")